# 🔥 S6E4 - BALANCED ACCURACY BEAST
### Optimized for Balanced Accuracy | Multi-Model Ensemble

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

print('='*70)
print('🔥 S6E4 - BALANCED ACCURACY BEAST')
print('='*70)

In [ ]:
# Load data
print('\n[1/6] Loading data...')
train = pd.read_csv('/kaggle/input/playground-series-s6e4/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s6e4/test.csv')

print(f'  Train: {train.shape}, Test: {test.shape}')
print(f'  Target distribution:')
print(f'  {train["Irrigation_Need"].value_counts().to_dict()}')

In [ ]:
# Advanced Feature Engineering
print('\n[2/6] Feature engineering...')

train['is_train'] = True
test['is_train'] = False
test['Irrigation_Need'] = 'Low'

df = pd.concat([train, test], axis=0, ignore_index=True)

# Encode categoricals
cat_cols = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 
            'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# Interaction features (domain knowledge)
df['Soil_pH_Soil_Moisture'] = df['Soil_pH'] * df['Soil_Moisture']
df['Temperature_Humidity'] = df['Temperature_C'] * df['Humidity']
df['Rainfall_Soil_Moisture'] = df['Rainfall_mm'] * df['Soil_Moisture']
df['Sunlight_Temperature'] = df['Sunlight_Hours'] * df['Temperature_C']
df['Wind_Humidity'] = df['Wind_Speed_kmh'] * df['Humidity']
df['Organic_Temperature'] = df['Organic_Carbon'] * df['Temperature_C']
df['EC_Moisture'] = df['Electrical_Conductivity'] * df['Soil_Moisture']

# Ratio features
df['Organic_to_Soil'] = df['Organic_Carbon'] / (df['Soil_Moisture'] + 1e-6)
df['EC_to_Soil_pH'] = df['Electrical_Conductivity'] / (df['Soil_pH'] + 1e-6)
df['Rainfall_per_Hectare'] = df['Rainfall_mm'] / (df['Field_Area_hectare'] + 1e-6)
df['Prev_Irrigation_per_Hectare'] = df['Previous_Irrigation_mm'] / (df['Field_Area_hectare'] + 1e-6)
df['Temp_Humidity_Ratio'] = df['Temperature_C'] / (df['Humidity'] + 1e-6)
df['Rainfall_Humidity'] = df['Rainfall_mm'] / (df['Humidity'] + 1e-6)

# Polynomial features
df['Soil_pH_sq'] = df['Soil_pH'] ** 2
df['Moisture_sq'] = df['Soil_Moisture'] ** 2
df['Temperature_sq'] = df['Temperature_C'] ** 2
df['Humidity_sq'] = df['Humidity'] ** 2
df['Rainfall_log'] = np.log1p(df['Rainfall_mm'])
df['Prev_Irrigation_log'] = np.log1p(df['Previous_Irrigation_mm'])

# Domain-specific features
df['Water_Stress'] = (df['Temperature_C'] * df['Wind_Speed_kmh']) / (df['Humidity'] + df['Soil_Moisture'] + 1e-6)
df['Irrigation_Efficiency'] = df['Previous_Irrigation_mm'] / (df['Rainfall_mm'] + 1e-6)
df['Crop_Water_Demand'] = df['Sunlight_Hours'] * df['Temperature_C'] / (df['Humidity'] + 1e-6)
df['Evapotranspiration_Proxy'] = (df['Temperature_C'] * df['Sunlight_Hours'] * df['Wind_Speed_kmh']) / (df['Humidity'] + 1e-6)
df['Soil_Water_Retention'] = df['Soil_Moisture'] * df['Organic_Carbon'] / (df['Electrical_Conductivity'] + 1e-6)

# Binned features
df['Soil_Moisture_bin'] = pd.qcut(df['Soil_Moisture'], q=10, labels=False, duplicates='drop')
df['Rainfall_bin'] = pd.qcut(df['Rainfall_mm'], q=10, labels=False, duplicates='drop')
df['Temperature_bin'] = pd.qcut(df['Temperature_C'], q=10, labels=False, duplicates='drop')
df['Humidity_bin'] = pd.qcut(df['Humidity'], q=10, labels=False, duplicates='drop')

# Target encoding with multiple statistics
for group_col in ['Crop_Type', 'Soil_Type', 'Region', 'Season', 'Irrigation_Type']:
    train_mask = df['is_train'] == True
    
    # Mean encoding
    group_means = df[train_mask].groupby(group_col)['Irrigation_Need'].apply(
        lambda x: (x == 'High').mean() * 2 + (x == 'Medium').mean()
    )
    df[f'{group_col}_target_mean'] = df[group_col].map(group_means)
    
    # Count encoding
    group_counts = df[train_mask].groupby(group_col).size()
    df[f'{group_col}_count'] = df[group_col].map(group_counts)
    
    # High probability
    group_high = df[train_mask].groupby(group_col)['Irrigation_Need'].apply(
        lambda x: (x == 'High').mean()
    )
    df[f'{group_col}_high_prob'] = df[group_col].map(group_high)

# Cross features between key categorical variables
df['Crop_Soil'] = df['Crop_Type'].astype(str) + '_' + df['Soil_Type'].astype(str)
df['Crop_Region'] = df['Crop_Type'].astype(str) + '_' + df['Region'].astype(str)
df['Crop_Season'] = df['Crop_Type'].astype(str) + '_' + df['Season'].astype(str)
df['Soil_Region'] = df['Soil_Type'].astype(str) + '_' + df['Region'].astype(str)

cross_cols = ['Crop_Soil', 'Crop_Region', 'Crop_Season', 'Soil_Region']
for col in cross_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

# Split back
train_final = df.iloc[:len(train)].copy()
test_final = df.iloc[len(train):].copy()
train_final['Irrigation_Need'] = train['Irrigation_Need']

train_final.drop(['is_train', 'id'], axis=1, inplace=True)
test_final.drop(['is_train'], axis=1, inplace=True)
if 'Irrigation_Need' in test_final.columns:
    test_final.drop(['Irrigation_Need'], axis=1, inplace=True)
if 'id' in test_final.columns:
    test_final.drop(['id'], axis=1, inplace=True)

print(f'  Total features: {train_final.shape[1] - 1}')

In [ ]:
# Prepare data
print('\n[3/6] Preparing...')

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(train_final['Irrigation_Need'])
X = train_final.drop('Irrigation_Need', axis=1)
X_test = test_final.copy()

print(f'  Classes: {target_encoder.classes_}')
print(f'  Features: {X.shape[1]}')
print(f'  Class distribution: {np.bincount(y)}')

In [ ]:
# Train models with CV - Optimized for BALANCED accuracy
print('\n[4/6] Training ensemble...')

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Store OOF and test predictions
oof_preds = np.zeros((len(X), 3))  # 3 models
test_preds = np.zeros((len(X_test), 3, 3))  # 3 classes, 3 models

seeds = [42, 123, 456, 789, 2024]

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f'\n  Fold {fold+1}/{N_SPLITS}')
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # Calculate class weights for balanced learning
    from sklearn.utils.class_weight import compute_sample_weight
    sample_weights = compute_sample_weight('balanced', y_train)
    
    # Model 1: XGBoost
    xgb_model = xgb.XGBClassifier(
        n_estimators=1500, max_depth=6, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        gamma=0.2, reg_alpha=0.5, reg_lambda=2.0,
        scale_pos_weight=1, random_state=seeds[fold], 
        eval_metric='mlogloss', tree_method='hist', verbosity=0
    )
    xgb_model.fit(X_train, y_train, sample_weight=sample_weights,
                 eval_set=[(X_val, y_val)], verbose=False)
    xgb_pred = xgb_model.predict(X_val)
    oof_preds[val_idx, 0] = xgb_pred
    test_preds[:, 0, :] += xgb_model.predict_proba(X_test) / N_SPLITS
    
    bal_acc_xgb = balanced_accuracy_score(y_val, xgb_pred)
    print(f'    XGB balanced_acc: {bal_acc_xgb:.6f}')
    
    # Model 2: LightGBM
    lgb_model = lgb.LGBMClassifier(
        n_estimators=1500, max_depth=6, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, min_child_samples=15,
        reg_alpha=0.5, reg_lambda=2.0, class_weight='balanced',
        random_state=seeds[fold], verbosity=-1
    )
    lgb_model.fit(X_train, y_train,
                  eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(100, verbose=False), 
                            lgb.log_evaluation(0)])
    lgb_pred = lgb_model.predict(X_val)
    oof_preds[val_idx, 1] = lgb_pred
    test_preds[:, 1, :] += lgb_model.predict_proba(X_test) / N_SPLITS
    
    bal_acc_lgb = balanced_accuracy_score(y_val, lgb_pred)
    print(f'    LGBM balanced_acc: {bal_acc_lgb:.6f}')
    
    # Model 3: CatBoost
    cat_model = CatBoostClassifier(
        iterations=1500, depth=6, learning_rate=0.03,
        l2_leaf_reg=2.0, random_seed=seeds[fold], 
        verbose=0, task_type='CPU', auto_class_weights='Balanced'
    )
    cat_model.fit(X_train, y_train, eval_set=(X_val, y_val),
                  early_stopping_rounds=100, verbose=False)
    cat_pred = cat_model.predict(X_val).flatten()
    oof_preds[val_idx, 2] = cat_pred
    test_preds[:, 2, :] += cat_model.predict_proba(X_test) / N_SPLITS
    
    bal_acc_cat = balanced_accuracy_score(y_val, cat_pred)
    print(f'    CAT balanced_acc: {bal_acc_cat:.6f}')

# Overall OOF scores
print(f'\n  Overall OOF Balanced Accuracies:')
for i, name in enumerate(['XGB', 'LGBM', 'CAT']):
    bal_acc = balanced_accuracy_score(y, oof_preds[:, i])
    print(f'    {name}: {bal_acc:.6f}')

In [ ]:
# Ensemble with weighted probabilities
print('\n[5/6] Creating ensemble...')

# Try different weights for each model based on OOF performance
model_scores = []
for i, name in enumerate(['XGB', 'LGBM', 'CAT']):
    score = balanced_accuracy_score(y, oof_preds[:, i])
    model_scores.append(score)

# Normalize weights
weights = np.array(model_scores) / sum(model_scores)
print(f'  Model weights: {weights}')

# Weighted ensemble
test_ensemble = np.zeros((len(X_test), 3))
for i in range(3):
    test_ensemble += weights[i] * test_preds[:, i, :]

# Also try simple average
test_simple_avg = np.mean([test_preds[:, i, :] for i in range(3)], axis=0)

# Use the one with higher confidence
test_pred_weighted = np.argmax(test_ensemble, axis=1)
test_pred_simple = np.argmax(test_simple_avg, axis=1)

print(f'  Weighted avg distribution: {pd.Series(test_pred_weighted).value_counts().to_dict()}')
print(f'  Simple avg distribution: {pd.Series(test_pred_simple).value_counts().to_dict()}')

In [ ]:
# Create submission
print('\n[6/6] Creating submission...')

# Use simple average (usually more robust)
test_pred = test_pred_simple
test_labels = target_encoder.inverse_transform(test_pred)

submission = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': test_labels
})

print(f'\n  Final predictions:')
print(f'  {submission["Irrigation_Need"].value_counts().to_dict()}')

submission.to_csv('submission.csv', index=False)
print('\n  ✓ submission.csv saved!')
print('\n' + '='*70)
print('🔥 DONE! Submit now and crush the leaderboard!')
print('='*70)